# Week 1 - Research Notebook
## Variant Effect Predictor for nicotinic acetylcholine receptors (nAChRs)

*Date: 2026-06-13*

**What this document is.** A written research brief answering the four Week-1 questions from the advisor:

1. Why do **positional** and **subunit** features matter? *(literature survey)*
2. What are **RDKit** and **Mordred**, do they fit our problem, and what libraries might fit better? *(ML feature libraries)*
3. What is an **ablation study**?
4. **Structural features** - what did the advisor mean by "we only need one (open or closed)", why do the current ones matter, and what could we add?

**How to read it.** Each section first explains the idea in plain language ("what it is / why it matters"), then connects it to *our* model, and ends with concrete suggestions. Citations appear inline as `[Author Year]` and are collected at the end. No code is run here - this is a reading and decision document.

> **Reminder of what we already have** (so the suggestions below are grounded): a binary classifier that predicts whether a missense variant is **Loss-of-Function (LOF)** or **Gain-of-Function (GOF)**, trained on **351 curated human nAChR variants** using **52 features** in four groups - physicochemical (24), substitution (3), positional (17), structural (8). Best model so far is logistic regression at **F1 ~= 0.66**.


## 0. Orientation - the receptor and our prediction task

### 0.1 What an nAChR is (in one paragraph)
A nicotinic acetylcholine receptor is a **pentamer**: five protein subunits arranged like the staves of a barrel around a central **ion channel**. It is a *ligand-gated ion channel* - when the neurotransmitter **acetylcholine (ACh)** (or nicotine) binds in pockets on the outside, the channel **opens** and lets cations (Na+, K+, Ca2+) flow through, which is how a nerve or muscle signal is passed on. Remove the agonist and it closes again. (Reviews: [Changeux 2012], [Gotti & Clementi 2004], [Zoli 2015].)

### 0.2 The two things a variant's "address" tells us
Every variant in our data is described by **which subunit** it sits in and **which position** in that subunit's sequence. Those two facts alone carry a lot of signal, because the receptor has a strict architecture (Section 1) and because different subunits build different receptors with different jobs (Section 2).

### 0.3 What we are predicting: LOF vs GOF
- **Loss-of-Function (LOF):** the variant makes the receptor work *less* - weaker binding, less opening, less current, or the receptor never reaches the cell surface.
- **Gain-of-Function (GOF):** the variant makes it work *more* - opens too easily, stays open too long, or leaks.

| Fact about our dataset | Value |
|---|---|
| Variants (after cleaning) | 351 |
| LOF / GOF | 218 / 133 |
| Subunits represented | 15 |
| Most common subunits | CHRNA1 (104), CHRNE (65), CHRNA7 (40) |
| Best model so far | Logistic regression, F1 ~= 0.66 +/- 0.01 |

> **Why this framing matters for everything below:** as Section 3 shows, LOF vs GOF is, mechanistically, largely a question of **which way a variant tips the balance between the closed and open states of the channel**. Keep that sentence in mind - it is the thread that ties the structural features, and the advisor's "open or closed" comment, together.


## 1. Positional features - *where* in the protein the variant sits

### 1.1 What the feature is in our code
Right now "position" is a **single number**: `position_normalized` = residue position divided by a length constant, squashed to roughly 0-1 (`vep_nachr/features/encoder.py`). That is the entire positional signal apart from the subunit one-hot (Section 2).

### 1.2 Why position matters: the receptor is built in distinct zones
Each subunit is one chain that folds into the same layered architecture. Reading from the start of the mature protein, top (synapse) to bottom (cytoplasm):

```
 Synapse (outside the cell)
   |
   |  [ Signal peptide ]              cleaved off; not in the mature protein
   |  [ Extracellular domain (ECD) ]  ACh binding site at subunit interfaces;
   |    loops A-F, C-loop, Cys-loop   agonist recognition + start of gating
   |  -- ECD / TMD interface --       beta1-beta2 loop, Cys-loop, pre-M1, M2-M3 linker
 Membrane
   |  [ M1 ]
   |  [ M2 ]   <-- lines the ion pore; the gate is here (the 9' leucine)
   |  [ M3 ]
   |  [ M3-M4 intracellular loop ]    long, variable; trafficking + modulation
   |  [ M4 ]   <-- faces the lipid
   |
 Cytoplasm (inside the cell)
```

The functional consequence of a mutation depends heavily on **which zone** it lands in:

| Zone (region) | What happens there | Typical effect of a disruptive variant |
|---|---|---|
| Orthosteric site / C-loop (ECD interface) | ACh/nicotine recognition; the C-loop "clamps" the agonist | Changes agonist affinity/efficacy -> LOF (or GOF if it stabilises activation) |
| ECD-TMD interface (beta1-beta2 loop, Cys-loop, M2-M3 linker) | Mechanically couples binding to channel opening ("gating") | Gating shifts -> LOF or GOF |
| M2 pore lining (esp. the **9' leucine**) | Forms the gate and narrowest part of the pore | Classic **GOF / slow-channel** when the gate is destabilised [Labarca 1995] |
| M1/M3/M4, lipid-facing | Helix packing, lipid interface | Folding/assembly defects -> often LOF |
| M3-M4 intracellular loop | Trafficking, phosphorylation, conductance | Surface-expression / modulation changes |

### 1.3 The single most famous example: the 9' leucine
Deep in the M2 pore helix there is a leucine conserved across essentially all subunits, at the position labelled **9'** (M2 residues are numbered with primes from the cytoplasmic end). Mutating it lowers the energy needed to open the channel, so the receptor opens at lower agonist concentrations and stays open longer - a textbook **gain-of-function** change, and the molecular basis of several **slow-channel congenital myasthenic syndromes** [Labarca 1995; Engel 2015]. The lesson for us: *the same chemical change (say L->S) means very different things at position 9' of M2 versus a random surface loop.*

### 1.4 What this means for our feature
A single normalised number **cannot** express "this residue is in the C-loop" or "this is a pore-lining M2 residue". Worse, position 250 means a different zone in different subunits (their domains start and end at different places), so the raw normalised position **mixes zones across subunits**. Sensible upgrades (for later, not now):
- a **categorical region label** (signal peptide / ECD / binding-site loop / Cys-loop / M2 / other TM / intracellular loop), derived from each subunit's UniProt topology;
- distance-based geometric features (Section 3): distance to the pore axis, distance to the ACh site.

> **Take-away:** position is a *proxy for functional zone*. It already helps the model, but its current one-number form is the crudest possible version; the biology says a region/zone encoding would capture far more.


## 2. Subunit features - *which* subunit the variant is in

### 2.1 What the feature is in our code
A **one-hot** vector over 16 human subunits (`subunit_CHRNA1` ... `subunit_CHRNG`): exactly one entry is 1, the rest 0 (`vep_nachr/features/encoder.py`; subunit list in `vep_nachr/config.py`).

### 2.2 The subunit family map
Humans have **16 nAChR subunit genes** that fall into groups building different receptors:

| Group | Subunits | Builds | Where / job | Disease link (in our data) |
|---|---|---|---|---|
| Muscle | alpha1, beta1, delta, epsilon, gamma | (alpha1)2-beta1-delta-epsilon (adult); ...-gamma (fetal) | Neuromuscular junction - muscle contraction | **CMS** (slow/fast-channel myasthenia) |
| Neuronal alpha | alpha2-alpha7, alpha9, alpha10 | heteromers + homomeric alpha7, alpha9-alpha10 | Brain, autonomic, cochlea | ADNFLE (alpha4), schizophrenia (alpha7), pain (alpha9-alpha10) |
| Neuronal beta | beta2, beta3, beta4 | partner the alphas | Brain, autonomic ganglia | ADNFLE (beta2), nicotine dependence (alpha4-beta2, alpha3-beta4) |

Key receptor subtypes worth knowing:
- **(alpha4)2(beta2)3 and (alpha4)3(beta2)2** - the main nicotine-binding receptor in the brain; mutations cause **ADNFLE epilepsy** and it is central to **nicotine dependence** [Zoli 2015].
- **alpha7 homopentamer** - five identical alpha7 subunits; fast-desensitising, highly Ca2+ permeable; linked to **schizophrenia / sensory gating** [Freedman 1997].
- **alpha3-beta4** - dominant in **autonomic ganglia**; the relay between CNS and periphery.
- **Muscle receptor** - the target in **congenital myasthenic syndromes (CMS)**, where slow-channel (GOF) and fast-channel (LOF) mutations are well catalogued [Engel 2015].

### 2.3 Principal vs complementary face (why "alpha" is special)
The ACh site sits at the **interface of two neighbouring subunits**: one contributes the **principal (+) face** (with the C-loop and its hallmark vicinal cysteines - the reason these subunits are called "alpha"), the other contributes the **complementary (-) face**.

```
   subunit A (alpha)              subunit B
   principal (+) face    |    complementary (-) face
            C-loop  -->  [ ACh binding pocket ]
```

So which subunits are present, and *in what order around the ring*, sets how many binding sites there are and what they prefer. Subunit identity is therefore not a cosmetic label - it determines pharmacology and kinetics.

### 2.4 Why one-hot subunit features help our model
- Different subunits have **different baseline behaviour** (kinetics, desensitisation, Ca2+ flux), so the *prior probability* of a variant being GOF vs LOF differs by subunit.
- The **same position** can mean different things in different subunits.
- Subunit correlates with **disease context** (the Pathology column), which correlates with how each effect was defined.

### 2.5 Three cautions our data forces on us
1. **Imbalance:** CHRNA1 (104) and CHRNE (65) - both muscle - make up nearly half the data, reflecting the rich CMS literature. The model may simply learn "muscle behaves like X". Report **per-subunit** performance, not only overall F1.
2. **Leakage risk:** if variants from the same subunit land in both train and test folds, one-hot subunit features let the model "memorise the subunit". Consider **subunit- or structure-grouped cross-validation** so generalisation to *unseen* subunits is tested honestly.
3. **Loses similarity:** one-hot treats alpha3 and alpha4 as no more alike than alpha3 and delta. A **family grouping** or a subunit-level property vector could encode real similarity.

> **Take-away:** subunit identity packs pharmacology, stoichiometry, and disease context into one label. One-hot is a reasonable start; the open issues are data imbalance and honest cross-validation, not the encoding itself.


## 3. Structural features - and decoding "we only need one: open or closed"

This is the section the advisor said the most (and the least clear) about, so we will go slowly.

### 3.1 A channel is a machine with a few discrete shapes ("states")
A nAChR is not static. It cycles through at least three **conformational states**:

```
        agonist binds                  prolonged agonist
  RESTING / CLOSED  --------->  OPEN  -------------------->  DESENSITISED
   (no agonist,               (agonist bound,              (agonist bound,
    gate shut,                 gate open,                    gate shut again,
    non-conducting)            CONDUCTING)                   non-conducting)
        ^                                                         |
        |________________________ agonist leaves _________________|
```

- **Resting / closed:** the everyday "off" state - no agonist, no current.
- **Open:** agonist bound, the gate is dilated, ions flow. This is the *functional* state.
- **Desensitised:** agonist still bound but the gate has shut again - a protective "off" state after prolonged exposure.

(Reviews of the gating cycle: [Noviello 2021], [Changeux 2012], [Nemecz 2016].)

### 3.2 Why this is the heart of LOF vs GOF
Most disease variants do not break the protein outright - they **re-balance these states**:
- **GOF** = tips the equilibrium *toward open* (opens too easily / stays open). e.g. slow-channel CMS, many M2 pore mutations.
- **LOF** = tips it *toward closed/desensitised* (will not open, or opens too briefly), or stops the receptor reaching the surface. e.g. fast-channel CMS.

So the question "is this variant GOF or LOF?" is, mechanistically, **"which way does it move the closed-to-open balance?"** That single sentence is why the advisor keeps coming back to *open vs closed*.


### 3.3 What "we only need one structural feature (open or closed)" most likely means
A residue only matters for *gating* if its surroundings **change between the closed and open shapes**. A residue that looks identical in both states is not part of the moving machinery; a residue that is buried in one state and exposed in the other (or that gains/loses neighbours) sits right at the heart of the open-to-closed switch - exactly where GOF/LOF gating mutations cluster (the C-loop, M2, the M2-M3 linker, the beta1-beta2 and Cys-loops).

So the *one* structural feature that best captures the LOF/GOF mechanism is a **difference-between-states feature**: compute a structural property in the **closed** structure and again in the **open** structure, and feed the model the **change**. The most natural choice is:

> **delta-RSA = (solvent exposure when open) - (solvent exposure when closed)**, optionally alongside delta-(C-beta density) or C-alpha displacement between the two states.

A residue with a large delta-RSA is, by definition, part of the gating motion - a strong prior that a mutation there will be GOF or LOF rather than silent.

**There are two readings of the advisor's words - confirm which one with the advisor (see Section 6):**
1. **(Most likely, and most powerful):** add **one** new feature that encodes the *open-minus-closed* structural difference per residue. This is a genuinely new, mechanism-aware feature.
2. **(Simpler reading):** make sure every structural feature we already compute is **labelled with the state it came from** - i.e. add a single "is this from an open or a closed structure?" flag - because right now we do not track that at all (see the problem below).

### 3.4 A real problem this exposes in our current pipeline
Checking the structures our code actually uses (`vep_nachr/config.py` -> `PDB_MAPPING`), they are **in different conformational states**:

| Subunits | PDB used | Conformational state |
|---|---|---|
| Muscle: alpha1, beta1, delta, epsilon, gamma | **7QKO** (Torpedo muscle nAChR) | **Resting / closed** [Zarkadas 2022] |
| alpha7 | **7EKI** | **Apo / resting closed** |
| alpha4, beta2 | **6CNJ** | alpha4-beta2 heteromer [Walsh 2018] - *confirm exact state* |
| alpha3, beta4 | **6PV7** | **Nicotine-bound** (agonist; desensitised-like) [Gharpure 2019] |

So our 8 structural features are computed from a **mixture of states** - mostly closed, but alpha3-beta4 comes from an *agonist-bound* structure. The model is being fed an "RSA" that means subtly different things for different subunits. **This is almost certainly the methodological gap the advisor is pointing at.** Fixing it (use the same state everywhere, and/or add the open-minus-closed difference) is higher value than adding more raw structural columns.

### 3.5 Good news: the open/closed comparison is actually feasible
The alpha7 receptor has a **matched set of structures in all three states** from one study [Noviello 2021]:

| State | alpha7 PDB |
|---|---|
| Resting / closed | 7KOO |
| Activated / open | 7KOX |
| Desensitised | 7KOQ |

That means we can compute a real **delta-RSA(open - closed)** for alpha7 today, and the muscle and alpha4-beta2 / alpha3-beta4 systems have resting + agonist-bound structures we can pair up with more effort. Subunits with no structure fall back to the existing imputation.


### 3.6 Why each of the 8 current structural features is meaningful
(from `vep_nachr/features/structural.py`)

| Feature | Plain meaning | Why it predicts variant effect |
|---|---|---|
| `rsa` (relative solvent accessibility) | How exposed vs buried the residue is (0 buried -> 1 exposed) | Buried-core mutations destabilise folding/assembly -> often LOF; surface mutations are usually better tolerated [Tien 2013] |
| `bfactor` | How mobile/"wobbly" the atom is in the structure | Flexible functional loops behave differently from the rigid core; flags mobile machinery |
| `dssp_helix` / `dssp_sheet` / `dssp_coil` | Local secondary structure (helix / sheet / loop) [Kabsch 1983] | Context matters: e.g. a proline forced into a helix is far more damaging than in a loop |
| `cbeta_density` | How many other residues pack within ~10 Angstrom | Tightly packed positions tolerate size changes poorly (steric clashes) |
| `hse_up` / `hse_down` | Half-sphere exposure: neighbours on the "up" vs "down" side of the residue [Hamelryck 2005] | A *directional* exposure measure - captures "buried toward the pore" vs "exposed to lipid" better than RSA alone |

These are all **per-residue, single-structure** descriptors - solid, standard, and cheap. Their limitation is exactly the theme of this section: each describes **one snapshot**, so on its own it cannot say whether a residue is part of the open-to-closed motion.

### 3.7 Candidate new structural features (ranked)
**Tier 1 - nAChR-specific geometry (highest value, mechanism-aware):**
1. **delta-RSA (open - closed)** and/or C-alpha displacement between states - the gating feature from Section 3.3.
2. **Radial distance to the pore axis** - separates pore-lining residues (gate/conductance) from outer residues. Unique to the pentamer's barrel geometry.
3. **Axial position along the membrane normal** - where along the channel (ECD vs mid-membrane vs intracellular).
4. **Distance to the nearest subunit-subunit interface** - interface residues affect assembly and the binding site.
5. **Distance to the orthosteric (ACh) site** - proximity to where agonist binds.
6. **Pore-facing flag / M2 "prime" number** - explicitly marks gate residues like 9'.

**Tier 2 - generic but useful:** residue depth, raw contact number, evolutionary conservation / coevolution (from an MSA), packing defects, lipid-facing flag, proximity to known Ca2+ sites.

**Practical note:** several of these are correlated (radial distance, pore-facing, and delta-RSA all light up the pore), so add a few and let the **ablation study** (Section 5) decide which pay off. Given our 24% structural-imputation gap, **adding AlphaFold models for the 6 subunits with no experimental structure (CHRNA2, CHRNA5, CHRNA6, CHRNA9, CHRNB3)** would likely help more than any single new column, because it turns imputed rows into real measurements.


## 4. ML feature libraries - RDKit, Mordred ("modrit"), and what might fit better

### 4.1 First, an honest framing
Our task is a **protein** problem (a mutation changes one amino acid in a big protein). RDKit and Mordred are **small-molecule chemistry** tools - built to describe *drug-like molecules*, not proteins. So we have to be a little clever about how, and whether, they fit. Let us take each in turn.

### 4.2 RDKit - what it is
**RDKit** is the standard open-source **cheminformatics** toolkit [Landrum]. Give it a molecule as a SMILES string (e.g. ethanol = `CCO`) and it can:
- compute ~200 **molecular descriptors** (molecular weight, logP, H-bond donor/acceptor counts, topological indices, ...),
- generate **fingerprints** (Morgan/ECFP, MACCS) - bit-vectors describing substructures,
- build 3-D conformers, do substructure search, and more.

It is the engine most other descriptor tools sit on top of.

### 4.3 Mordred ("modrit") - what it is
**Mordred** is a **molecular descriptor calculator built on top of RDKit** [Moriwaki 2018]. Its selling point is breadth: **~1,800 descriptors** (2-D and some 3-D) in one call - all of RDKit's plus many more (constitutional, topological, autocorrelation, charge, ring descriptors, ...).
- Practical note: the original `mordred` package is old and can clash with new Python/NumPy; there is a maintained community fork (commonly `mordred-community`) to install instead. Verify at install time.

### 4.4 The key question: how do small-molecule descriptors help a *protein* model?
The trick - and almost certainly what the advisor means by "look into RDKit/Mordred" - is to treat **each of the 20 amino acids as a small molecule**. Every amino acid (or just its side chain) has a SMILES string, so you can:

1. write the 20 SMILES once (glycine `C(C(=O)O)N`, alanine `CC(C(=O)O)N`, ...),
2. run RDKit/Mordred to get a large descriptor vector for each amino acid,
3. for a variant W->M, build features from the **wild-type vector, the mutant vector, and their difference** - exactly the wt/mt/delta pattern we already use for the 8 hand-picked AAindex scales, but now with hundreds of chemically grounded descriptors instead of 8.

So Mordred would be a **drop-in enrichment of our physicochemical block** - same idea, much richer chemistry.

**Caveats (important at our small sample size):**
- Over just 20 molecules many descriptors are **constant or near-duplicate** -> must filter (drop zero-variance, drop highly correlated).
- ~1,800 columns on **351 samples** is a recipe for **overfitting** -> reduce hard (variance/correlation filters, PCA, or a curated subset) before modelling.
- These describe the **isolated amino acid**, not its protein context - so they complement, not replace, the structural features.

### 4.5 Libraries that fit a protein VEP *better*
Because ours is a protein/variant task, these are arguably a better use of effort than chemistry descriptors:

| Library / approach | What it gives you | Fit for us |
|---|---|---|
| **AAindex** (already used) | 500+ curated amino-acid property scales | Easy win: expand beyond our current 8 scales, or PCA them |
| **`peptides`** (Python) | QSAR descriptors, z-scales, VHSE for residues/windows | Good for features of the *sequence window* around the mutation |
| **iFeature / iFeatureOmega** | Many protein/peptide sequence descriptors + selection tools | Sequence-context features |
| **propy3 / BioPython ProtParam** | Composition + physicochemical sequence features | Lightweight, easy |
| **Protein language models - ESM-1v / ESM-2, ProtT5** | A learned "fluency" score for any sequence; the **wt->mut log-likelihood ratio is a zero-shot variant-effect score** | **Highest payoff** - state of the art for VEP [Meier 2021; Lin 2023] |
| **Pre-computed VEP scores - AlphaMissense, EVE, REVEL, PolyPhen-2, SIFT** | One pathogenicity/effect number per variant from huge models | Strong single features to add [Cheng 2023; Frazer 2021] |

### 4.6 Recommendation

| Priority | Add | Why | Effort |
|---|---|---|---|
| 1 | **ESM-1v / ESM-2 zero-shot score** (one number per variant) | Modern SOTA for variant effect; usually beats hand-built features; no training needed | Medium (run a pretrained model) |
| 2 | **Mordred amino-acid descriptors** (wt/mt/delta, filtered) | Directly answers the advisor; cheaply enriches the AA block | Low-medium |
| 3 | **Sequence-window descriptors** (`peptides` / iFeature) | Adds local context the per-residue features miss | Low |

> **Bottom line:** RDKit/Mordred *can* help us - by describing the 20 amino acids as molecules and enriching the physicochemical block - but they are small-molecule tools used at an angle. For a *variant* predictor the bigger, well-evidenced win is a **protein language model score (ESM)**. Recommend trying both and letting the ablation study decide.


## 5. What is an "ablation study"?

### 5.1 Plain definition
The word comes from experimental biology/medicine, where to "ablate" means to **remove** a piece (e.g. lesion a brain region) and see what stops working. In machine learning it means the same thing: **remove or switch off one part of your system, re-run, and measure how much performance drops.**
- If removing part X **hurts** performance -> X was contributing.
- If removing X **changes nothing** -> X was dead weight (and you can drop it). [Meyes 2019]

It is how you turn "we have 52 features" into "here is the evidence for which features actually earn their place."

### 5.2 Common kinds
| Kind | What you remove | Question it answers |
|---|---|---|
| **Leave-one-group-out** | one whole feature group at a time | "Does the *structural* block help at all?" |
| **Add-one-group-in** | start from nothing, add groups cumulatively | "How much does each group add on top of the others?" |
| **Single-feature** | one column at a time | "Is RSA specifically pulling its weight?" |
| **Component / step** | a preprocessing or model step (e.g. scaling, tuning) | "Is the scaler or the Optuna tuning worth it?" |

### 5.3 Why it matters *for us specifically*
- We have **four feature groups** (physicochemical / substitution / positional / structural). An ablation tells us which to invest in.
- We **already have a partial result**: in `NOTES2.ipynb`, switching the structural block on barely moved F1 (about 0.656 -> 0.659), within noise. **That is exactly an ablation finding** - it says the current structural block, as built, is not earning much yet, which is precisely why Section 3's *better* structural features matter.
- It is expected in any methods write-up or paper: reviewers will ask "what does each part contribute?"

### 5.4 How we could run one *without changing any code*
Our pipeline is already built for this:
- the feature encoder exposes switches **`include_structural`, `include_substitution`, `include_positional`** (`vep_nachr/features/encoder.py`);
- `vep_nachr/config.py` -> `FEATURE_GROUPS` lists the four groups and their sizes;
- the existing cross-validation in `vep_nachr/training/cross_validation.py` already reports F1.

So an ablation is just: *for each combination of switches, build the features, run the existing CV, and record mean +/- std F1.* A results table would look like:

| Feature groups used | # features | Mean F1 | delta vs full |
|---|---|---|---|
| All four (full model) | 52 | 0.66 | - |
| minus structural | 44 | ? | ? |
| minus substitution | 49 | ? | ? |
| minus positional | 35 | ? | ? |
| physicochemical only | 24 | ? | ? |

### 5.5 The one statistical caution
Our dataset is small (351), so F1 wobbles by **+/- 0.01 to 0.03** just from the random fold split. To avoid fooling ourselves:
- use the **same random seeds** for every ablation run (the code already uses 5 fixed seeds),
- report **mean +/- std**, not a single number,
- treat a change **smaller than the noise band as "no effect"**.

Also remember that **group ablation can hide redundancy** - if two groups carry the same signal, removing either one alone looks harmless. That is why you also do *add-one-in*.

> **Take-away:** an ablation study is the experiment that converts "we built these features" into "here is the evidence for which ones matter". Ours is essentially free to run because the on/off switches already exist.


## 6. Summary, recommendations, and questions to confirm with the advisor

### 6.1 The four answers in one line each
- **Positional features** encode *which functional zone* a residue is in (binding site, gating interface, M2 pore, ...). Effect depends on zone - the 9' leucine in M2 is the classic example. Our single normalised number is the crudest form; a region label would be much stronger.
- **Subunit features** encode *which receptor* and *which biology* - pharmacology, stoichiometry, and disease context all ride on subunit identity. One-hot is fine; the real issues are data imbalance (muscle-heavy) and using subunit-grouped CV to test generalisation honestly.
- **Structural features** describe the residue's physical environment. The advisor's "open or closed" points at the deepest truth of the project: **LOF vs GOF = which way the variant tips the closed-to-open balance**, so the most informative structural feature is the **open-minus-closed difference**. We also found that our current structures mix states - worth fixing.
- **Feature libraries:** RDKit/Mordred are small-molecule tools we can repurpose to enrich the amino-acid block; the bigger modern win for a variant predictor is a **protein language model score (ESM)**. An **ablation study** is how we prove which of all these features actually help.

### 6.2 Prioritised recommendations
1. **Make the structural state consistent** across subunits, and add **one open-minus-closed difference feature** (start with alpha7, which has matched open/closed/desensitised structures). *(Directly answers the advisor.)*
2. **Add a protein-language-model (ESM) zero-shot score** as a feature and benchmark it.
3. **Try Mordred amino-acid descriptors** (wt/mt/delta, heavily filtered) to enrich the physicochemical block. *(Directly answers the advisor.)*
4. **Run a proper ablation** over the four groups (and the new features) using the existing `include_*` switches.
5. **Add AlphaFold structures** for the 6 subunits with no experimental structure, to cut the 24% imputation - likely more impactful than any single new structural column.

### 6.3 Questions to confirm with the advisor
1. By "we only need one structural feature (open or closed)", do you mean **(a)** add a single *open-minus-closed difference* feature, or **(b)** just **tag** each existing structural feature with the state it came from? (This brief argues (a) is the stronger interpretation.)
2. For RDKit/Mordred - is the intent to **describe the 20 amino acids as molecules** and feed wt/mt/delta descriptors (my assumption), or something else (e.g. describing bound ligands like ACh/nicotine)?
3. Are we open to a **protein language model (ESM)** feature, or do you want to keep the feature set fully hand-engineered and interpretable for now?
4. Should cross-validation move to **subunit-grouped** folds to test generalisation to unseen subunits?


## References

*Citations anchor each claim; please verify exact details (author lists, volume/page) against the original sources before any formal submission. PDB conformational states quoted in Section 3 were checked against RCSB entries.*

**nAChR biology, structure, and gating**
- Changeux J-P (2012). The nicotinic acetylcholine receptor: the founding father of the pentameric ligand-gated ion channel superfamily. *J Biol Chem* 287:40207-40215.
- Gotti C, Clementi F (2004). Neuronal nicotinic receptors: from structure to pathology. *Prog Neurobiol* 74:363-396.
- Zoli M, Pistillo F, Gotti C (2015). Diversity of native nicotinic receptor subtypes in mammalian brain. *Neuropharmacology* 96:302-311.
- Nemecz A, Prevost MS, Menny A, Corringer P-J (2016). Emerging molecular mechanisms of signal transduction in pentameric ligand-gated ion channels. *Neuron* 90:452-470.

**Disease / functional variants**
- Labarca C, Nowak MW, Zhang H, et al. (1995). Channel gating governed symmetrically by conserved leucine residues in the M2 domain of nicotinic receptors. *Nature* 376:514-516.
- Engel AG, Shen X-M, Selcen D, Sine SM (2015). Congenital myasthenic syndromes: pathogenesis, diagnosis, and treatment. *Lancet Neurol* 14:420-434.
- Steinlein OK, Mulley JC, Propping P, et al. (1995). A missense mutation in the neuronal nicotinic acetylcholine receptor alpha4 subunit is associated with autosomal dominant nocturnal frontal lobe epilepsy. *Nat Genet* 11:201-203.
- Freedman R, Coon H, Myles-Worsley M, et al. (1997). Linkage of a neurophysiological deficit in schizophrenia to a chromosome 15 locus. *PNAS* 94:587-592.

**nAChR structures (and the PDB IDs our code uses)**
- Noviello CM, Gharpure A, Mukhtasimova N, et al. (2021). Structure and gating mechanism of the alpha7 nicotinic acetylcholine receptor. *Cell* 184:2121-2134. (alpha7 states: 7KOO resting, 7KOX activated/open, 7KOQ desensitised.)
- Walsh RM Jr, Roh S-H, Gharpure A, et al. (2018). Structural principles of distinct assemblies of the human alpha4beta2 nicotinic receptor. *Nature* 557:261-265. (PDB 6CNJ.)
- Morales-Perez CL, Noviello CM, Hibbs RE (2016). X-ray structure of the human alpha4beta2 nicotinic receptor. *Nature* 538:411-415. (first alpha4beta2; PDB 5KXI.)
- Gharpure A, Teng J, Zhuang Y, et al. (2019). Agonist selectivity and ion permeation in the alpha3beta4 ganglionic nicotinic receptor. *Neuron* 104:501-511. (PDB 6PV7.)
- Zarkadas E, Pebay-Peyroula E, Thompson MJ, et al. (2022). Conformational transitions and ligand-binding to a muscle-type nicotinic acetylcholine receptor. *Neuron* 110:1358-1370. (Torpedo muscle nAChR; PDB 7QKO resting.)

**Structural descriptors used in our features**
- Kabsch W, Sander C (1983). Dictionary of protein secondary structure (DSSP). *Biopolymers* 22:2577-2637.
- Tien MZ, Meyer AG, Sydykova DK, Spielman SJ, Wilke CO (2013). Maximum allowed solvent accessibilities of residues in proteins. *PLoS ONE* 8:e80635.
- Hamelryck T (2005). An amino acid has two sides: a new 2D measure provides a different view of solvent exposure. *Proteins* 59:38-48.
- Grantham R (1974). Amino acid difference formula to help explain protein evolution. *Science* 185:862-864.
- Henikoff S, Henikoff JG (1992). Amino acid substitution matrices from protein blocks (BLOSUM). *PNAS* 89:10915-10919.

**Feature libraries and modern VEP methods**
- Landrum G. RDKit: Open-source cheminformatics. https://www.rdkit.org
- Moriwaki H, Tian Y-S, Kawashita N, Takagi T (2018). Mordred: a molecular descriptor calculator. *J Cheminform* 10:4.
- Meier J, Rao R, Verkuil R, Liu J, Sercu T, Rives A (2021). Language models enable zero-shot prediction of the effects of mutations on protein function (ESM-1v). *NeurIPS* / bioRxiv 2021.07.09.450648.
- Lin Z, Akin H, Rao R, et al. (2023). Evolutionary-scale prediction of atomic-level protein structure with a language model (ESM-2 / ESMFold). *Science* 379:1123-1130.
- Frazer J, Notin P, Dias M, et al. (2021). Disease variant prediction with deep generative models of evolutionary data (EVE). *Nature* 599:91-95.
- Cheng J, Novati G, Pan J, et al. (2023). Accurate proteome-wide missense variant effect prediction with AlphaMissense. *Science* 381:eadg7492.
- Chen Z, Zhao P, Li F, et al. (2018). iFeature: a Python package and web server for features extraction and selection from protein and peptide sequences. *Bioinformatics* 34:2499-2502.
- Osorio D, Rondon-Villarreal P, Torres R (2015). Peptides: a package for data mining of antimicrobial peptides. *R Journal* 7:4-14.

**Methodology**
- Meyes R, Lu M, de Puiseau CW, Meisen T (2019). Ablation studies in artificial neural networks. *arXiv:1901.08644*.
